# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

**Некрасов Артур Дмитрович**

Профіль T4 завантажує оригінальні SmolLM3 у fp16. CPU-профіль використовує Q4_K_M цих самих моделей і ті самі токенізатори. Час генерації залежить від профілю. Моделі завантажуються почергово.

In [ ]:
%%time
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("pip", 'install -q "transformers==5.20.2" "datasets==5.1.0" "accelerate==1.15.0"')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
CPU times: user 3.27 s, sys: 739 ms, total: 4.01 s
Wall time: 40.6 s


In [ ]:
%%time
import atexit
import gc
import json
import os
import platform
import socket
import subprocess
import tarfile
import time
import urllib.request
from pathlib import Path

import pandas as pd
import torch
from datasets import get_dataset_split_names, load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
CPU_PROFILE = device == "cpu"
if CPU_PROFILE:
    torch.set_num_threads(4)
    print("CPU-профіль: SmolLM3-3B Base/Instruct у GGUF Q4_K_M")
else:
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Пам’ять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU: Tesla T4
Пам’ять GPU: 15.64 GB
CPU times: user 19 s, sys: 2.04 s, total: 21.1 s
Wall time: 42.5 s


In [ ]:
%%time
if device == "cuda":
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout)
else:
    print("Генерація виконується на CPU.")

Wed Nov 11 19:48:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [ ]:
%%time
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"
base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

gguf_sources = {
    base_model_name: ("mradermacher/SmolLM3-3B-Base-GGUF", "SmolLM3-3B-Base.Q4_K_M.gguf", "f164d390684f654824a1ab12b860d6b4ba6cbbad"),
    instruct_model_name: ("unsloth/SmolLM3-3B-GGUF", "SmolLM3-3B-Q4_K_M.gguf", "a7bc17204c8a326d6bd6e466e076959eddae2025"),
}

def runtime_binary():
    provided = os.environ.get("LAB5_LLAMA_SERVER")
    if provided:
        return Path(provided).resolve()
    if platform.system() != "Linux" or platform.machine() not in {"x86_64", "AMD64"}:
        raise RuntimeError("Для демо 2 у Windows відкрий ноутбук у Colab і вибери T4 GPU.")
    folder = Path("lab5_cpu_runtime").resolve()
    binary = folder / "llama-b12110" / "llama-server"
    if binary.is_file():
        return binary
    folder.mkdir(exist_ok=True)
    filename = "llama-b12110-bin-ubuntu-x64.tar.gz"
    archive = folder / filename
    urllib.request.urlretrieve(f"https://github.com/ggml-org/llama.cpp/releases/download/b12110/{filename}", archive)
    with tarfile.open(archive) as bundle:
        bundle.extractall(folder, filter="data")
    return binary

class ModelSession:
    def __init__(self, model_id):
        self.model_id = model_id
        self.model = None
        self.process = None
        self.url = None
        self.opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
        atexit.register(self.close)

    def open(self):
        if not CPU_PROFILE:
            if self.model is None:
                self.model = AutoModelForCausalLM.from_pretrained(self.model_id, dtype=torch.float16, device_map=device)
                self.model.eval()
            return
        if self.process is not None:
            return
        repo, filename, revision = gguf_sources[self.model_id]
        assets = os.environ.get("LAB5_GGUF_DIR")
        model_file = Path(assets) / filename if assets else None
        if model_file is None or not model_file.is_file():
            model_file = Path(hf_hub_download(repo, filename, revision=revision))
        with socket.socket() as listener:
            listener.bind(("127.0.0.1", 0))
            port = listener.getsockname()[1]
        self.url = f"http://127.0.0.1:{port}"
        log_file = Path(self.model_id.split("/")[-1] + "-gguf.log")
        command = [str(runtime_binary()), "-m", str(model_file), "--host", "127.0.0.1", "--port", str(port),
            "--n-gpu-layers", "0", "--threads", "4", "--parallel", "1", "--ctx-size", "2048",
            "--cache-ram", "0", "--no-cache-prompt", "--no-webui"]
        with log_file.open("w", encoding="utf-8") as log:
            self.process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT)
        deadline = time.monotonic() + 90
        while time.monotonic() < deadline:
            if self.process.poll() is not None:
                break
            try:
                with self.opener.open(self.url + "/health", timeout=2) as response:
                    if response.status == 200:
                        return
            except OSError:
                time.sleep(0.25)
        self.close()
        raise RuntimeError(f"llama-server не запустився. Журнал: {log_file}")

    def close(self):
        if self.process is not None:
            if self.process.poll() is None:
                self.process.terminate()
                try:
                    self.process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    self.process.kill()
                    self.process.wait()
            self.process = None
        if self.model is not None:
            self.model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

base_model = ModelSession(base_model_name)
instruct_model = ModelSession(instruct_model_name)
print("Токенізатори завантажено. Ваги завантажуються почергово під час генерації.")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

Токенізатори завантажено. Ваги завантажуються почергово під час генерації.
CPU times: user 4.99 s, sys: 314 ms, total: 5.31 s
Wall time: 27.8 s


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [ ]:
%%time
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [ ]:
%%time
generation_log = []

def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    model.open()
    started = time.perf_counter()
    if CPU_PROFILE:
        payload = {"prompt": text, "n_predict": max_new_tokens, "temperature": temperature, "seed": 42,
            "stream": False, "cache_prompt": False, "stop": ["<|im_end|>"]}
        request = urllib.request.Request(model.url + "/completion", data=json.dumps(payload).encode(),
            headers={"Content-Type": "application/json"})
        with model.opener.open(request, timeout=300) as response:
            result = json.load(response)
        answer = result["content"]
        generated = int(result.get("tokens_predicted", len(tokenizer.encode(answer, add_special_tokens=False))))
        stop_reason = result.get("stop_type", "")
    else:
        inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.model.device)
        with torch.inference_mode():
            output = model.model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature,
                do_sample=True, pad_token_id=tokenizer.eos_token_id)
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
        generated = len(new_tokens)
        stop_reason = "limit" if generated >= max_new_tokens else "stop"
    elapsed = time.perf_counter() - started
    generation_log.append({"model": model.model_id, "tokens": generated, "seconds": elapsed, "stop": stop_reason})
    return answer.strip()

CPU times: user 9 µs, sys: 0 ns, total: 9 µs
Wall time: 13.1 µs


In [ ]:
%%time
test_prompt = "Explain me, what is quantum computing?"
base_answer = generate(base_model, base_tokenizer, test_prompt)
base_model.close()
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(instruct_messages, tokenize=False, add_generation_prompt=True)
instruct_answer = generate(instruct_model, instruct_tokenizer, instruct_formatted)
print("BASE:\n" + base_answer)
print("\nINSTRUCT:\n" + instruct_answer)
display(pd.DataFrame(generation_log).round(3))

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


... (4 outputs left out)

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

BASE:
What's the difference from classic computing?
You can think of a quantum computer as a computer with quantum bits instead of binary bits. In a binary computer, information is stored in a series of switches, where only two values are possible: either switched on or off. A binary computer has to store and process information in the form of a sequence of "on" or "off" bits, i.e. ones and zeroes. Such a system is called a digital computer.
A quantum computer is a device that stores information in the form of quantum bits, or qubits. In a quantum computer, a qubit can be a spin of an electron, or a photon, or any other physical system that can be represented by a two-level system. The

INSTRUCT:
Quantum computing is a type of computing that utilizes the principles of quantum mechanics to perform calculations and solve complex problems. Unlike classical computers, which use bits (0s and 1s) to store and process information, quantum computers use quantum bits, or qubits, which can exist

,model,tokens,seconds,stop
0,HuggingFaceTB/SmolLM3-3B-Base,150,7.827,limit
1,HuggingFaceTB/SmolLM3-3B,150,6.659,limit


CPU times: user 1min 8s, sys: 36.7 s, total: 1min 45s
Wall time: 3min 38s


## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [ ]:
%%time
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]
reference_answers = ["360", "5 cups", "$18.95"]
reasoning_rows = []
for flag in ["/no_think", "/think"]:
    for index, prompt in enumerate(reasoning_prompts):
        messages = [{"role": "system", "content": flag}, {"role": "user", "content": prompt}]
        formatted = instruct_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        generated = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        if "</think>" in generated:
            final = generated.rsplit("</think>", 1)[1].strip()
        elif flag == "/think" and "<think>" in formatted and not formatted.rstrip().endswith("</think>"):
            final = ""
        else:
            final = generated
        row = {"режим": flag, "задача": index + 1, "еталон": reference_answers[index],
            "відповідь": final, "токени": generation_log[-1]["tokens"],
            "секунди": generation_log[-1]["seconds"], "зупинка": generation_log[-1]["stop"]}
        reasoning_rows.append(row)
        print(f"{flag}, задача {index + 1}, еталон {reference_answers[index]}")
        print(final if final else "Фінальна відповідь не з’явилася в межах бюджету.")
        print(f"Токени: {row['токени']}, час: {row['секунди']:.2f} с\n")
instruct_model.close()
display(pd.DataFrame(reasoning_rows).drop(columns="відповідь").round(3))

/no_think, задача 1, еталон 360
To find the product of 15 and 24, we can use the standard multiplication algorithm. Here's how to do it step by step:

First, write down the numbers vertically, aligning the rightmost digits:

```
   15
×  24
```

Next, multiply 15 by 4 (the rightmost digit of 24):

```
   15
×  24
------
  60  (15 × 4)
```

Then, multiply 15 by 20 (the leftmost digit of 24, shifted one position to the left):

```
   15
×  24
------
  60  (15 × 4)
120  (15 × 20)
------
```

Now, add the two partial products:

```
   15
×  24
------
  60  (15 × 4)
120  (15 × 20)
------
  360
```

So, the product of 15 and 24 is 360.
Токени: 215, час: 8.42 с

/no_think, задача 2, еталон 5 cups
To determine how much flour is needed for 30 cookies, we can start by finding the amount of flour required for one cookie and then scale it up to 30 cookies.

First, we know that 2 cups of flour are needed for 12 cookies. To find out how much flour is needed for one cookie, we divide the total amount

,режим,задача,еталон,токени,секунди,зупинка
0,/no_think,1,360,215,8.422,stop
1,/no_think,2,5 cups,155,6.356,stop
2,/no_think,3,$18.95,119,4.643,stop
3,/think,1,360,300,17.878,limit
4,/think,2,5 cups,300,12.237,limit
5,/think,3,$18.95,300,12.324,limit


CPU times: user 58.1 s, sys: 64 ms, total: 58.1 s
Wall time: 1min 2s


## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [ ]:
%%time
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

template_examples = {}
for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    model_tokenizer = AutoTokenizer.from_pretrained(name)
    template_examples[name] = model_tokenizer.apply_chat_template(messages, tokenize=False)

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>

CPU times: user 5.32 s, sys: 275 ms, total: 5.6 s
Wall time: 17.4 s


Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [ ]:
%%time
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>


CPU times: user 409 ms, sys: 19.8 ms, total: 429 ms
Wall time: 1.38 s


### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [ ]:
%%time
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True
CPU times: user 143 µs, sys: 1e+03 ns, total: 144 µs
Wall time: 132 µs


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [ ]:
%%time
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [ ]:
%%time
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
for message in everyday[0]["messages"]:
    content = message["content"] or ""
    if message["role"] == "assistant" and "</think>" in content:
        reasoning, content = content.split("</think>", 1)
        print(f"[assistant] Блок think: {len(reasoning)} символів перед закривальним тегом.")
    print(f"[{message['role']}] {content.strip()[:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})
[user] Hi there
[assistant] Блок think: 429 символів перед закривальним тегом.
[assistant] Hello! How can I assist you today?
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] Блок think: 1777 символів перед закривальним тегом.
[assistant] Here’s a quick and nutritious breakfast idea that’s easy to customize:

### **Avocado Toast with a Twist**  
**Ingredients:**  
- 1 slice whole-grain or sourdough bread (toasted)  
- ½ ripe avocado, m
CPU times: user 186 ms, sys: 98 ms, total: 284 ms
Wall time: 3.43 s


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [ ]:
%%time
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'role': 'system', 'content': 'You are a math tutor. Solve problems step by step.'}, {'role': 'user', 'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}, {'role': 'assistant', 'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}]}
CPU times: user 271 ms, sys: 74.8 ms, total: 346 ms
Wall time: 5.67 s


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [ ]:
%%time
text = instruct_tokenizer.apply_chat_template(gsm8k_messages[0]["messages"],
    tokenize=False, add_generation_prompt=False)
print(text)
formatted_ids = instruct_tokenizer(text, add_special_tokens=False)["input_ids"]
print(f"Довжина в токенах: {len(formatted_ids)}")
measurements = {"device": device, "profile": "GGUF Q4_K_M" if CPU_PROFILE else "fp16",
    "generation": generation_log, "base_answer": base_answer, "instruct_answer": instruct_answer,
    "reasoning": reasoning_rows, "templates": template_examples, "smoltalk_splits": splits,
    "everyday_size": len(everyday), "gsm8k_tokens": len(formatted_ids)}
Path("chat_demo_measurements.json").write_text(json.dumps(measurements, ensure_ascii=False, indent=2), encoding="utf-8")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145
CPU times: user 2.06 ms, sys: 1.02 ms, total: 3.08 ms
Wall time: 4.01 ms


7347

**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

## 0.1. Відповіді на питання 11–21

### 11. Час клітинок

На Tesla T4 сума `Wall time` для 15 клітинок становила близько 7 хв 03 с. Завантаження токенізаторів тривало 27,8 с, порівняння Base та Instruct разом із завантаженням моделей — 3 хв 38 с, а перевірка математичних задач — 1 хв 02 с. `pip` повідомив про конфлікт `pyarrow` і `cudf-cu13`. Усі клітинки завершилися успішно.

### 12. `add_generation_prompt`

Під час інференсу `True` додає початок нового ходу асистента, тому модель продовжує діалог із потрібної ролі. Для завершеного діалогу з еталонною відповіддю в SFT використовують `False`. У секції 2 різницю видно в кінці відформатованого входу. З `True` після останнього повідомлення з’являється заголовок `assistant` і, залежно від режиму, початок блоку міркувань. За самим текстом відповіді прапорець однозначно відновити важко, тому порівнюють повні серіалізовані входи.

### 13. Ролі `system` і `assistant`

`system` задає глобальний контекст, стиль, обмеження й параметри діалогу. `assistant` містить попередні відповіді моделі або цільову відповідь у тренувальних прикладах. Заголовок `assistant` у кінці входу позначає місце, звідки слід почати генерацію.

### 14. Форматування в секції 2

У виводі SmolLM3 шаблон додає `## Metadata` з `Knowledge Cutoff Date`, поточною датою та `Reasoning Mode`. У `## Custom Instructions` міститься системний текст. Кожен хід має маркер `<|im_start|>` із роллю та межу `<|im_end|>`. У режимі `/no_think` після заголовка нового ходу є порожній закритий блок `<think>…</think>`. Для багатокрокового діалогу зберігається чергування `user` і `assistant`. Прапорець `add_generation_prompt` керує заголовком наступної відповіді. Ці маркери допомагають моделі розрізняти інструкції, текст користувача й історію відповідей.

### 15. Base та Instruct

Base отримала запит як звичайний текст і продовжила його запитанням «What's the difference from classic computing?». Instruct отримала запит через шаблон діалогу, одразу дала визначення квантових обчислень і почала нумерований перелік понять. Це відповідає їхньому навчанню. Base вчилася продовжувати текст, а Instruct додатково вчилася відповідати на інструкції. Обидві генерації досягли ліміту 150 токенів.

### 16. Бюджет `max_new_tokens=300`

Ліміт 300 поширюється на всю генерацію, включно з проміжним міркуванням і фінальною відповіддю. Він дає більше місця, ніж 150 токенів у простому порівнянні, і водночас обмежує час. В обох режимах використано однаковий ліміт. Якщо модель витратить його на блок `think`, фінальний висновок може обірватися або не з’явитися.

### 17. Арифметичні задачі

Режим `/no_think` дав три правильні фінальні відповіді — 360, 5 чашок борошна і 18,95 долара. На них витрачено 215, 155 і 119 токенів відповідно. У першій задачі був неправильний проміжний крок. Модель вказала, що 15 × 20 дорівнює 120, хоча правильний добуток — 300.

У режимі `/think` усі три генерації досягли ліміту 300 токенів до появи фінальної відповіді. Для цих задач за такого ліміту я б обрав `/no_think` і перевіряв проміжні обчислення.

### 18. Шаблони різних моделей

У секції 5 один діалог серіалізовано трьома токенізаторами. SmolLM3 додає системні метадані й використовує межі `<|im_start|>` / `<|im_end|>`. Qwen3 має такі самі маркери ходів, але в цьому прикладі системний текст подано без блоку метаданих, а в ході асистента видно `<think>…</think>`. Phi-3.5 використовує окремі маркери `<|system|>`, `<|user|>`, `<|assistant|>`, завершує ходи `<|end|>` і додає `<|endoftext|>` наприкінці. Для Qwen3 `enable_thinking=False` впливає на префікс відповіді. Конкретний набір токенів потрібно брати з шаблону тієї моделі, яку використовуємо.

### 19. Шаблон у SFT

SFT вчить модель продовжувати послідовність, отриману після серіалізації `messages`. Формат навчання має узгоджуватися з форматом інференсу. Правильні межі ходів потрібні для визначення токенів відповіді асистента та маскування loss. Чужий шаблон або зайвий заголовок після завершеного діалогу змінюють послідовність, на якій навчається модель.

### 20. Джерела SmolTalk2

SmolTalk2 складається з `Mid`, `SFT` і `Preference`. Для середнього етапу використано Llama-Nemotron Post-Training Dataset і OpenThoughts3. У суміші SFT є SmolTalk, OpenThoughts3, LongAlign, Aya, Table-GPT, приклади викликів інструментів та інші компоненти. Частину відповідей згенеровано на наявних промптах за допомогою Qwen3-32B. Приклади поділено на `think` і `no_think`. Preference спирається на Tulu 3 8B Preference Mixture і створені на його запитах пари для режиму міркувань. У виводі є 25 частин SFT. Показаний піднабір повсякденних діалогів містить 2057 рядків. [Картка SmolTalk2](https://huggingface.co/datasets/HuggingFaceTB/smoltalk2)

### 21. Перетворення GSM8K у `messages`

Код бере перші 100 прикладів train. Для кожної пари створює системну інструкцію математичного наставника, переносить `question` у `user` і весь `answer` у `assistant`. Після `map` вихідні колонки замінено колонкою `messages`. Шаблон перетворює ролі й тексти на послідовність токенів для завершеного діалогу. У першому прикладі з цією конфігурацією отримано 145 токенів.